In [19]:
# ============================================================================
# ZOMATO RESTAURANT CLUSTERING & SENTIMENT ANALYSIS
# ============================================================================

# Basic Libraries
import os
import json
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warning control
import warnings
warnings.filterwarnings("ignore")

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# NLP - Sentiment Analysis
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Visualization for text
from wordcloud import WordCloud

# Utility libraries
import re
import joblib
from datetime import datetime

In [20]:
# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'RESTAURANT_CSV': r"/content/Zomato Restaurant names and Metadata.csv",
    'REVIEWS_CSV': r"/content/Zomato Restaurant reviews.csv",
    'OUTPUT_DIR': "output_artifacts",
    'N_CLUSTERS': 5,
    'RANDOM_STATE': 42,
    'MIN_REVIEWS': 5,
}

# Create output folder
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

print("Configuration Loaded Successfully")
print(CONFIG)

Configuration Loaded Successfully
{'RESTAURANT_CSV': '/content/Zomato Restaurant names and Metadata.csv', 'REVIEWS_CSV': '/content/Zomato Restaurant reviews.csv', 'OUTPUT_DIR': 'output_artifacts', 'N_CLUSTERS': 5, 'RANDOM_STATE': 42, 'MIN_REVIEWS': 5}


In [21]:
# Download NLTK resources
import nltk

resources = ['vader_lexicon', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except:
        print(f"Could not download {resource}")

In [22]:
# ============================================================================
# PART 1: DATA LOADING
# ============================================================================

print("="*80)
print("ZOMATO RESTAURANT ANALYSIS - MULTI-ALGORITHM CLUSTERING")
print("="*80)

def load_and_inspect_data():
    """Load restaurant metadata and review data."""
    print("\n[1/10] Loading Data...")

    try:
        # Show available files (useful in Colab)
        print("\nAvailable files in working directory:")
        print(os.listdir())

        # Check file paths
        if not os.path.exists(CONFIG['RESTAURANT_CSV']):
            raise FileNotFoundError(f"Restaurant file not found: {CONFIG['RESTAURANT_CSV']}")

        if not os.path.exists(CONFIG['REVIEWS_CSV']):
            raise FileNotFoundError(f"Reviews file not found: {CONFIG['REVIEWS_CSV']}")

        # Load CSV files
        df_meta = pd.read_csv(CONFIG['RESTAURANT_CSV'], encoding='utf-8', low_memory=False)
        df_reviews = pd.read_csv(CONFIG['REVIEWS_CSV'], encoding='utf-8', low_memory=False)

        # Dataset size
        print(f"\n✓ Metadata Loaded: {df_meta.shape[0]:,} rows, {df_meta.shape[1]} columns")
        print(f"✓ Reviews Loaded: {df_reviews.shape[0]:,} rows, {df_reviews.shape[1]} columns")

        # Column names
        print("\nMetadata Columns:")
        print(df_meta.columns.tolist())

        print("\nReview Columns:")
        print(df_reviews.columns.tolist())

        # Preview data
        print("\nSample Metadata:")
        print(df_meta.head(3))

        print("\nSample Reviews:")
        print(df_reviews.head(3))

        return df_meta, df_reviews

    except Exception as e:
        print(f"\n❌ ERROR while loading data: {e}")
        raise

# Run loader
df_meta, df_reviews = load_and_inspect_data()

ZOMATO RESTAURANT ANALYSIS - MULTI-ALGORITHM CLUSTERING

[1/10] Loading Data...

Available files in working directory:
['.config', 'Zomato Restaurant reviews.csv', 'Zomato Restaurant names and Metadata.csv', 'output_artifacts', 'sample_data']

✓ Metadata Loaded: 105 rows, 6 columns
✓ Reviews Loaded: 10,000 rows, 7 columns

Metadata Columns:
['Name', 'Links', 'Cost', 'Collections', 'Cuisines', 'Timings']

Review Columns:
['Restaurant', 'Reviewer', 'Review', 'Rating', 'Metadata', 'Time', 'Pictures']

Sample Metadata:
              Name                                              Links   Cost  \
0  Beyond Flavours  https://www.zomato.com/hyderabad/beyond-flavou...    800   
1         Paradise  https://www.zomato.com/hyderabad/paradise-gach...    800   
2         Flechazo  https://www.zomato.com/hyderabad/flechazo-gach...  1,300   

                                         Collections  \
0  Food Hygiene Rated Restaurants in Hyderabad, C...   
1                                Hyderabad's H

In [23]:
# ============================================================================
# PART 2: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================================

def comprehensive_eda(meta, reviews, output_dir):
    """Perform comprehensive exploratory data analysis"""

    print("\n[2/10] Performing Exploratory Data Analysis...")

    # ------------------------------
    # Missing Values
    # ------------------------------

    print("\nMissing Values in Metadata:")
    missing_meta = (meta.isnull().sum() / len(meta) * 100).round(2)
    print(missing_meta[missing_meta > 0])

    print("\nMissing Values in Reviews:")
    missing_reviews = (reviews.isnull().sum() / len(reviews) * 100).round(2)
    print(missing_reviews[missing_reviews > 0])

    # ------------------------------
    # Clean Cost Column
    # ------------------------------

    if 'Cost' in meta.columns:
        meta['Cost_clean'] = meta['Cost'].astype(str).replace('[^0-9.]', '', regex=True)
        meta['Cost_clean'] = pd.to_numeric(meta['Cost_clean'], errors='coerce')

    # ------------------------------
    # Visualization
    # ------------------------------

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Exploratory Data Analysis - Zomato Restaurants', fontsize=16, fontweight='bold')

    # 1️⃣ Cost Distribution
    if 'Cost_clean' in meta.columns:
        axes[0,0].hist(meta['Cost_clean'].dropna(), bins=40, color='#ff6b6b', edgecolor='black')
        axes[0,0].set_title('Cost Distribution')
        axes[0,0].set_xlabel('Cost (₹)')
        axes[0,0].set_ylabel('Frequency')

        median_cost = meta['Cost_clean'].median()
        axes[0,0].axvline(median_cost, color='red', linestyle='--', label=f'Median ₹{median_cost:.0f}')
        axes[0,0].legend()

    # 2️⃣ Top Cuisines
    if 'Cuisines' in meta.columns:
        cuisines = meta['Cuisines'].fillna("").str.split(",").explode().str.strip()
        top_cuisines = cuisines.value_counts().head(10)

        axes[0,1].barh(top_cuisines.index[::-1], top_cuisines.values[::-1], color='#4ecdc4')
        axes[0,1].set_title('Top Cuisines')
        axes[0,1].set_xlabel('Number of Restaurants')

    # 3️⃣ Rating Distribution
    if 'Rating' in reviews.columns:
        reviews['Rating'] = pd.to_numeric(reviews['Rating'], errors='coerce')

        axes[1,0].hist(reviews['Rating'].dropna(), bins=20, color='#95e1d3', edgecolor='black')
        axes[1,0].set_title('Rating Distribution')
        axes[1,0].set_xlabel('Rating')
        axes[1,0].set_ylabel('Frequency')

    # 4️⃣ Reviews per Restaurant
    if 'Restaurant' in reviews.columns:
        review_counts = reviews['Restaurant'].value_counts()

        axes[1,1].hist(review_counts.values, bins=40, color='#ffeaa7', edgecolor='black')
        axes[1,1].set_title('Reviews per Restaurant')
        axes[1,1].set_xlabel('Number of Reviews')
        axes[1,1].set_ylabel('Number of Restaurants')
        axes[1,1].set_yscale('log')

    plt.tight_layout()

    # Save figure
    save_path = os.path.join(output_dir, "01_eda_overview.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"✓ EDA visualizations saved → {save_path}")

    return meta, reviews


# Run EDA
df_meta, df_reviews = comprehensive_eda(df_meta, df_reviews, CONFIG['OUTPUT_DIR'])


[2/10] Performing Exploratory Data Analysis...

Missing Values in Metadata:
Collections    51.43
Timings         0.95
dtype: float64

Missing Values in Reviews:
Reviewer    0.38
Review      0.45
Rating      0.38
Metadata    0.38
Time        0.38
dtype: float64
✓ EDA visualizations saved → output_artifacts/01_eda_overview.png


In [24]:
# ============================================================================
# PART 3: DATA CLEANING
# ============================================================================

def clean_and_prepare_data(meta, reviews):
    """Clean data and handle missing values intelligently"""

    print("\n[3/10] Cleaning Data & Handling Missing Values...")

    # ---------------------------------------------------
    # Clean column names
    # ---------------------------------------------------

    meta.columns = [c.strip() for c in meta.columns]
    reviews.columns = [c.strip() for c in reviews.columns]

    # ---------------------------------------------------
    # Remove duplicates
    # ---------------------------------------------------

    meta = meta.drop_duplicates().reset_index(drop=True)
    reviews = reviews.drop_duplicates().reset_index(drop=True)

    # ---------------------------------------------------
    # Drop column with too many missing values
    # ---------------------------------------------------

    if 'Collections' in meta.columns:
        meta.drop(columns=['Collections'], inplace=True)

    # ---------------------------------------------------
    # Clean Cost column
    # ---------------------------------------------------

    if 'Cost' in meta.columns:
        meta['Cost'] = (
            meta['Cost']
            .astype(str)
            .str.replace(r"[^\d.]", "", regex=True)
        )

        meta['Cost'] = pd.to_numeric(meta['Cost'], errors='coerce')

    # ---------------------------------------------------
    # Fill missing Cost values using cuisine median
    # ---------------------------------------------------

    if 'Cuisines' in meta.columns:

        meta['cuisine_primary'] = (
            meta['Cuisines']
            .fillna("")
            .apply(lambda x: x.split(",")[0].strip())
        )

        cost_by_cuisine = meta.groupby('cuisine_primary')['Cost'].median()

        def fill_cost(row):
            if pd.notnull(row['Cost']):
                return row['Cost']

            cuisine = row['cuisine_primary']

            if cuisine in cost_by_cuisine:
                return cost_by_cuisine[cuisine]

            return meta['Cost'].median()

        meta['Cost'] = meta.apply(fill_cost, axis=1)

    # ---------------------------------------------------
    # Clean reviews dataset
    # ---------------------------------------------------

    if 'Rating' in reviews.columns:
        reviews['Rating'] = pd.to_numeric(reviews['Rating'], errors='coerce')

    if 'Review' in reviews.columns:
        reviews = reviews.dropna(subset=['Review'])

    # ---------------------------------------------------
    # Detect cost outliers
    # ---------------------------------------------------

    if 'Cost' in meta.columns:

        Q1 = meta['Cost'].quantile(0.25)
        Q3 = meta['Cost'].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR

        outliers = meta[(meta['Cost'] < lower) | (meta['Cost'] > upper)]

        print(f"✓ Identified {len(outliers)} cost outliers (kept for analysis)")

    print(f"✓ Cleaned Dataset: {len(meta)} restaurants | {len(reviews)} reviews")

    return meta, reviews


# Run cleaning
df_meta, df_reviews = clean_and_prepare_data(df_meta, df_reviews)


[3/10] Cleaning Data & Handling Missing Values...
✓ Identified 0 cost outliers (kept for analysis)
✓ Cleaned Dataset: 105 restaurants | 9955 reviews


In [25]:
# ============================================================================
# PART 4: SENTIMENT ANALYSIS
# ============================================================================

def perform_sentiment_analysis(reviews, output_dir):
    """Perform sentiment analysis using VADER"""

    print("\n[4/10] Performing Sentiment Analysis...")

    # Initialize VADER
    try:
        sid = SentimentIntensityAnalyzer()
    except:
        print("⚠️ VADER not available")
        reviews['sentiment_score'] = 0
        reviews['sentiment_label'] = "Neutral"
        return reviews

    # --------------------------------------------
    # Identify review text column
    # --------------------------------------------

    text_col = None
    for col in reviews.columns:
        if "review" in col.lower():
            text_col = col
            break

    if text_col is None:
        print("⚠️ Review text column not found")
        reviews['sentiment_score'] = 0
        reviews['sentiment_label'] = "Neutral"
        return reviews

    # Clean review text
    reviews[text_col] = reviews[text_col].fillna("").astype(str)

    # --------------------------------------------
    # Calculate sentiment score
    # --------------------------------------------

    reviews['sentiment_score'] = reviews[text_col].apply(
        lambda x: sid.polarity_scores(x)['compound']
    )

    # --------------------------------------------
    # Label sentiments
    # --------------------------------------------

    reviews['sentiment_label'] = pd.cut(
        reviews['sentiment_score'],
        bins=[-1, -0.05, 0.05, 1],
        labels=["Negative", "Neutral", "Positive"]
    )

    # --------------------------------------------
    # Convert rating to numeric
    # --------------------------------------------

    if 'Rating' in reviews.columns:
        reviews['Rating'] = pd.to_numeric(reviews['Rating'], errors='coerce')

    # --------------------------------------------
    # Visualization
    # --------------------------------------------

    fig, axes = plt.subplots(2,2, figsize=(14,10))
    fig.suptitle("Sentiment Analysis Overview", fontsize=16)

    # Sentiment distribution
    sentiment_counts = reviews['sentiment_label'].value_counts()

    axes[0,0].pie(
        sentiment_counts.values,
        labels=sentiment_counts.index,
        autopct="%1.1f%%",
        colors=["#ff6b6b","#ffd93d","#6bcf7f"]
    )

    axes[0,0].set_title("Sentiment Distribution")

    # Sentiment score histogram
    axes[0,1].hist(reviews['sentiment_score'], bins=40, color="#4ecdc4")

    axes[0,1].set_title("Sentiment Score Distribution")
    axes[0,1].set_xlabel("Score")

    # Sentiment vs Rating
    if 'Rating' in reviews.columns:

        sentiment_by_rating = reviews.groupby("Rating")["sentiment_score"].mean()

        axes[1,0].bar(
            sentiment_by_rating.index,
            sentiment_by_rating.values,
            color="#95e1d3"
        )

        axes[1,0].set_title("Average Sentiment by Rating")
        axes[1,0].set_xlabel("Rating")

    # Word Cloud for positive reviews
    positive_reviews = reviews[reviews['sentiment_score'] > 0.5][text_col]

    if len(positive_reviews) > 0:

        text = " ".join(positive_reviews)

        wc = WordCloud(width=800,height=400,background_color="white").generate(text)

        axes[1,1].imshow(wc)
        axes[1,1].axis("off")
        axes[1,1].set_title("Positive Review WordCloud")

    plt.tight_layout()

    save_path = os.path.join(output_dir,"02_sentiment_analysis.png")

    plt.savefig(save_path,dpi=300)
    plt.close()

    # --------------------------------------------
    # Summary
    # --------------------------------------------

    total = len(reviews)

    print("✓ Sentiment analysis completed")

    print(f"Positive: {(reviews['sentiment_label']=='Positive').sum()/total*100:.2f}%")
    print(f"Neutral : {(reviews['sentiment_label']=='Neutral').sum()/total*100:.2f}%")
    print(f"Negative: {(reviews['sentiment_label']=='Negative').sum()/total*100:.2f}%")

    return reviews


# Run Sentiment Analysis
df_reviews = perform_sentiment_analysis(df_reviews, CONFIG['OUTPUT_DIR'])


[4/10] Performing Sentiment Analysis...
✓ Sentiment analysis completed
Positive: 0.85%
Neutral : 98.81%
Negative: 0.33%


In [29]:
# ============================================================================
# PART 5: AGGREGATE TO RESTAURANT LEVEL
# ============================================================================

# Define helper BEFORE it's used, and at module level (not nested)
def _normalize_name(name):
    if pd.isnull(name):
        return ""
    name = str(name).lower()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name


def aggregate_to_restaurant_level(meta, reviews):

    print("\n[5/10] Aggregating Reviews to Restaurant Level...")

    meta = meta.copy()
    reviews = reviews.copy()

    meta.columns = [c.strip() for c in meta.columns]
    reviews.columns = [c.strip() for c in reviews.columns]

    # Detect restaurant name columns
    meta_name_cols = [c for c in meta.columns if 'name' in c.lower()]
    rev_name_cols = [c for c in reviews.columns if 'restaurant' in c.lower()]

    if not meta_name_cols or not rev_name_cols:
        print("⚠️ Restaurant name columns not found")
        meta['num_reviews'] = 0
        meta['avg_sentiment'] = 0
        return meta

    mcol = meta_name_cols[0]
    rcol = rev_name_cols[0]

    # Normalize names
    meta['_mkey'] = meta[mcol].apply(_normalize_name)
    reviews['_rkey'] = reviews[rcol].apply(_normalize_name)

    # Aggregate sentiment metrics
    agg = reviews.groupby('_rkey').agg(
        num_reviews=('sentiment_score','count'),
        avg_sentiment=('sentiment_score','mean')
    ).reset_index()

    agg = agg.rename(columns={'_rkey':'_mkey'})

    df = meta.merge(agg, how='left', on='_mkey')

    df['num_reviews'] = df['num_reviews'].fillna(0).astype(int)
    df['avg_sentiment'] = df['avg_sentiment'].fillna(0)

    # Positive / Negative sentiment %
    sentiment_stats = reviews.groupby('_rkey')['sentiment_score'].apply(
        lambda x: pd.Series({
            'pct_positive': (x > 0.05).mean(),
            'pct_negative': (x < -0.05).mean()
        })
    ).reset_index()

    sentiment_stats = sentiment_stats.rename(columns={'_rkey':'_mkey'})

    df = df.merge(sentiment_stats, how='left', on='_mkey')

    if 'pct_positive' not in df.columns:
        df['pct_positive'] = 0
    if 'pct_negative' not in df.columns:
        df['pct_negative'] = 0

    df['pct_positive'] = df['pct_positive'].fillna(0)
    df['pct_negative'] = df['pct_negative'].fillna(0)

    # Average rating
    if 'Rating' in reviews.columns:
        rating_stats = reviews.groupby('_rkey')['Rating'].mean().reset_index()
        rating_stats = rating_stats.rename(
            columns={'_rkey':'_mkey','Rating':'avg_rating'}
        )
        df = df.merge(rating_stats, how='left', on='_mkey')

    if 'avg_rating' in df.columns:
        df['avg_rating'] = df['avg_rating'].fillna(df['avg_rating'].median())

    # Filter restaurants with minimum reviews
    df = df[df['num_reviews'] >= CONFIG['MIN_REVIEWS']].reset_index(drop=True)

    print(f"✓ Aggregated to {len(df)} restaurants")

    return df


# Run aggregation
df = aggregate_to_restaurant_level(df_meta, df_reviews)


[5/10] Aggregating Reviews to Restaurant Level...
✓ Aggregated to 200 restaurants


In [30]:
# ============================================================================
# PART 6: FEATURE ENGINEERING
# ============================================================================

def engineer_features(df):

    print("\n[6/10] Engineering Features...")

    df = df.copy()

    # --------------------------------------------------
    # Cuisine Features
    # --------------------------------------------------

    if 'Cuisines' in df.columns:

        df['num_cuisines'] = df['Cuisines'].fillna("").apply(
            lambda s: len([c for c in str(s).split(",") if c.strip()])
        )

        all_cuisines = df['Cuisines'].fillna("").str.split(",").explode().str.strip()

        top_cuis = all_cuisines.value_counts().head(10).index.tolist()

        for cuisine in top_cuis:

            col = f"cuisine_{re.sub(r'\\W+', '_', cuisine.lower()).strip('_')}"

            df[col] = df['Cuisines'].fillna("").apply(
                lambda s: 1 if cuisine in str(s) else 0
            )

    else:

        df['num_cuisines'] = 1

    # --------------------------------------------------
    # Restaurant Timing Feature
    # --------------------------------------------------

    if 'Timings' in df.columns:

        df['is_open_late'] = df['Timings'].fillna("").str.lower().apply(
            lambda s: 1 if 'pm' in s and any(x in s for x in ['10:', '11:', '12:']) else 0
        )

    else:

        df['is_open_late'] = 0

    # --------------------------------------------------
    # Numeric Columns
    # --------------------------------------------------

    num_reviews = df.get('num_reviews', pd.Series(0, index=df.index)).astype(float)
    avg_rating = df.get('avg_rating', pd.Series(3.5, index=df.index)).astype(float)
    avg_sent = df.get('avg_sentiment', pd.Series(0.0, index=df.index)).astype(float)
    cost = df.get('Cost', pd.Series(0.0, index=df.index)).astype(float)

    # --------------------------------------------------
    # Derived Features
    # --------------------------------------------------

    df['price_per_star'] = cost / (avg_rating + 1e-6)

    df['sentiment_rating_gap'] = ((avg_sent + 1) / 2 * 5) - avg_rating

    df['popularity_score'] = num_reviews * avg_rating * (1 + avg_sent) / 2

    df['value_score'] = avg_rating / (np.log1p(cost) + 1e-6)

    df['review_engagement'] = num_reviews / (cost + 1)

    df['cuisine_diversity'] = df['num_cuisines'] * avg_rating

    cuisine_feature_count = len([c for c in df.columns if c.startswith("cuisine_")])

    print(f"✓ Engineered features (including {cuisine_feature_count} cuisine columns)")

    return df


# Run feature engineering
df = engineer_features(df)


[6/10] Engineering Features...
✓ Engineered features (including 12 cuisine columns)


In [31]:
# ============================================================================
# PART 7: PREPARING FEATURES FOR CLUSTERING
# ============================================================================

print("\n[7/10] Preparing Features for Clustering...")

# Feature selection
feature_cols = [
    'Cost',
    'num_reviews',
    'avg_sentiment',
    'pct_positive',
    'pct_negative',
    'num_cuisines',
    'price_per_star',
    'review_engagement',
    'popularity_score',
    'value_score',
    'cuisine_diversity',
    'is_open_late'
]

if 'avg_rating' in df.columns:
    feature_cols.insert(1, 'avg_rating')

cuisine_cols = [c for c in df.columns if c.startswith('cuisine_')]

feature_cols = [f for f in feature_cols if f in df.columns] + cuisine_cols

print(f"Selected {len(feature_cols)} features for clustering")

# Keep only numeric columns
X = df[feature_cols].select_dtypes(include=[np.number]).fillna(0)

print("Feature matrix shape:", X.shape)

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=0.95, random_state=CONFIG['RANDOM_STATE'])
X_pca = pca.fit_transform(X_scaled)

print("Original features:", X_scaled.shape[1])
print("PCA components:", X_pca.shape[1])


[7/10] Preparing Features for Clustering...
Selected 25 features for clustering
Feature matrix shape: (200, 24)
Original features: 24
PCA components: 14


In [32]:
# ============================================================================
# PART 8: KMEANS CLUSTERING
# ============================================================================

print("\n[8/10] KMeans Clustering...")

# ---------------------------------------
# Run KMeans
# ---------------------------------------

kmeans = KMeans(
    n_clusters=CONFIG['N_CLUSTERS'],
    random_state=CONFIG['RANDOM_STATE'],
    n_init=20
)

df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

# ---------------------------------------
# Evaluate Clustering
# ---------------------------------------

sil_score = silhouette_score(X_scaled, df['kmeans_cluster'])

print(f"KMeans Silhouette Score: {sil_score:.4f}")

# ---------------------------------------
# Save Model
# ---------------------------------------

joblib.dump(
    kmeans,
    os.path.join(CONFIG['OUTPUT_DIR'], "kmeans_model.pkl")
)

print("✓ KMeans clustering completed")


[8/10] KMeans Clustering...
KMeans Silhouette Score: 0.1585
✓ KMeans clustering completed


In [33]:
# ============================================================================
# PART 9: HIERARCHICAL CLUSTERING
# ============================================================================

print("\n[9/10] Hierarchical Clustering...")

# ---------------------------------------
# Run Agglomerative Clustering
# ---------------------------------------

hierarchical = AgglomerativeClustering(
    n_clusters=CONFIG['N_CLUSTERS'],
    linkage='ward'
)

df['hierarchical_cluster'] = hierarchical.fit_predict(X_scaled)

# ---------------------------------------
# Evaluate clustering
# ---------------------------------------

sil_score = silhouette_score(X_scaled, df['hierarchical_cluster'])

print(f"Hierarchical Silhouette Score: {sil_score:.4f}")

print("✓ Hierarchical clustering completed")


[9/10] Hierarchical Clustering...
Hierarchical Silhouette Score: 0.1471
✓ Hierarchical clustering completed


In [34]:
# ============================================================================
# PART 10: CLUSTER PROFILING
# ============================================================================

print("\n[10/10] Cluster Profiling...")

cluster_col = "kmeans_cluster"

# ---------------------------------------
# Create Cluster Summary
# ---------------------------------------

agg_config = {
    'Cost': ['mean','median'],
    'num_reviews': ['mean','median'],
    'avg_sentiment': 'mean',
    'pct_positive': 'mean',
    'popularity_score': 'mean',
    'value_score': 'mean'
}

if 'avg_rating' in df.columns:
    agg_config['avg_rating'] = 'mean'

cluster_profile = df.groupby(cluster_col).agg(agg_config)

cluster_profile.columns = ['_'.join(col).strip('_') for col in cluster_profile.columns]

cluster_profile = cluster_profile.reset_index()

print("\nCluster Profiles:")
print(cluster_profile)

# ---------------------------------------
# Save Results
# ---------------------------------------

cluster_profile.to_csv(
    os.path.join(CONFIG['OUTPUT_DIR'], "cluster_profiles.csv"),
    index=False
)

df.to_csv(
    os.path.join(CONFIG['OUTPUT_DIR'], "restaurants_clustered.csv"),
    index=False
)

# ---------------------------------------
# PCA Visualization
# ---------------------------------------

plt.figure(figsize=(10,7))

scatter = plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df[cluster_col],
    cmap="tab10",
    s=60,
    alpha=0.7
)

plt.colorbar(scatter,label="Cluster")

plt.title("Restaurant Clusters (PCA Projection)")

plt.xlabel("Principal Component 1")

plt.ylabel("Principal Component 2")

plt.grid(alpha=0.3)

plt.savefig(
    os.path.join(CONFIG['OUTPUT_DIR'],"03_pca_clusters.png"),
    dpi=300
)

plt.close()

print("✓ Cluster profiling completed")


[10/10] Cluster Profiling...

Cluster Profiles:
   kmeans_cluster    Cost_mean  Cost_median  num_reviews_mean  \
0               0  1016.666667       1000.0        100.000000   
1               1  1518.421053       1400.0        100.000000   
2               2   700.000000        700.0         99.920000   
3               3   410.869565        400.0         99.913043   
4               4   937.037037        800.0         98.481481   

   num_reviews_median  avg_sentiment_mean  pct_positive_mean  \
0               100.0            0.003701                0.0   
1               100.0            0.002500                0.0   
2               100.0            0.003258                0.0   
3               100.0            0.003536                0.0   
4               100.0            0.001370                0.0   

   popularity_score_mean  value_score_mean  avg_rating_mean  
0             189.957512          0.551263         3.785833  
1             189.217091          0.519521         

In [35]:
# ============================================================================
# SUMMARY
# ============================================================================

summary = {
    'analysis_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'total_restaurants': len(df),
    'total_reviews': len(df_reviews),
    'clusters_identified': CONFIG['N_CLUSTERS'],
    'avg_sentiment': float(df['avg_sentiment'].mean()),
    'avg_cost': float(df['Cost'].median())
}

if 'avg_rating' in df.columns:
    summary['avg_rating'] = float(df['avg_rating'].mean())

with open(os.path.join(CONFIG['OUTPUT_DIR'], 'analysis_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print(f"✅ All artifacts saved to: {CONFIG['OUTPUT_DIR']}/")
print(f"📊 Summary: {len(df)} restaurants, {CONFIG['N_CLUSTERS']} clusters")
print(f"📁 Files saved: {len(os.listdir(CONFIG['OUTPUT_DIR']))} artifacts")
print("\n🚀 Ready for deployment!")


ANALYSIS COMPLETE!
✅ All artifacts saved to: output_artifacts/
📊 Summary: 200 restaurants, 5 clusters
📁 Files saved: 7 artifacts

🚀 Ready for deployment!
